# FoodChain AI - Recommendation Engine

## Objective

Build the final supplier recommendation engine that combines:

- User GPS location
- Ingredient filtering
- Haversine distance
- Delivery radius filtering
- Feature scaling
- K-Means prediction
- Business-aware weighted ranking

This notebook produces the **Top 5 supplier recommendations** for a street food vendor.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from math import radians, sin, cos, sqrt, atan2

# Load assets
df = pd.read_csv("../data/supplier_clustered.csv")
kmeans = joblib.load("../models/kmeans_model.pkl")
scaler = joblib.load("../models/scaler.pkl")

df.head()


## Haversine Distance Function

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)

    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2*atan2(sqrt(a), sqrt(1-a))

    return R*c


## Vendor Input (This will come from the frontend later)

In [ ]:
vendor = {
    "latitude":18.5204,
    "longitude":73.8567,
    "ingredient":"Potato"
}

vendor


## Filter Suppliers By Ingredient

In [ ]:
filtered = df[
    df["ingredient"].str.lower() ==
    vendor["ingredient"].lower()
].copy()

print("Available Suppliers:", len(filtered))
filtered.head()


## Calculate Distance

In [ ]:
filtered["distance_km"] = filtered.apply(
    lambda row: haversine_distance(
        vendor["latitude"],
        vendor["longitude"],
        row["latitude"],
        row["longitude"]
    ),
    axis=1
)

filtered[["supplier_name","distance_km"]].head()


## Apply Delivery Radius Filter

In [ ]:
filtered = filtered[
    filtered["distance_km"] <= filtered["delivery_radius_km"]
].copy()

print("Eligible Suppliers:", len(filtered))


## Prepare Features For Prediction

In [ ]:
features = [
    "price",
    "distance_km",
    "rating",
    "quality_score",
    "reliability_score",
    "average_delivery_time_min"
]

X = scaler.transform(filtered[features])

filtered["predicted_cluster"] = kmeans.predict(X)

filtered.head()


## Weighted Business Ranking

In [ ]:
# Normalize selected metrics
ranking = filtered.copy()

ranking["distance_score"] = 1 - (ranking["distance_km"] / ranking["distance_km"].max())
ranking["price_score"] = 1 - (ranking["price"] / ranking["price"].max())
ranking["rating_score"] = ranking["rating"] / 5
ranking["quality_score_norm"] = ranking["quality_score"] / 5
ranking["reliability_score_norm"] = ranking["reliability_score"] / 100
ranking["delivery_score"] = 1 - (ranking["average_delivery_time_min"] / ranking["average_delivery_time_min"].max())

ranking["recommendation_score"] = (
    0.35 * ranking["distance_score"] +
    0.25 * ranking["price_score"] +
    0.15 * ranking["rating_score"] +
    0.10 * ranking["quality_score_norm"] +
    0.10 * ranking["reliability_score_norm"] +
    0.05 * ranking["delivery_score"]
)

ranking = ranking.sort_values(
    "recommendation_score",
    ascending=False
)

ranking.head(10)


## Generate Top 5 Recommendations

In [ ]:
top5 = ranking[[
    "supplier_name",
    "area",
    "ingredient",
    "price",
    "distance_km",
    "rating",
    "quality_score",
    "average_delivery_time_min",
    "recommendation_score"
]].head(5)

top5.reset_index(drop=True, inplace=True)
top5


## Recommendation Function

In [ ]:
def recommend_supplier(latitude, longitude, ingredient, top_n=5):

    temp = df[df["ingredient"].str.lower()==ingredient.lower()].copy()

    temp["distance_km"] = temp.apply(
        lambda row: haversine_distance(
            latitude,
            longitude,
            row["latitude"],
            row["longitude"]
        ),
        axis=1
    )

    temp = temp[
        temp["distance_km"] <= temp["delivery_radius_km"]
    ].copy()

    X = scaler.transform(temp[features])

    temp["cluster"] = kmeans.predict(X)

    temp["distance_score"] = 1-(temp["distance_km"]/temp["distance_km"].max())
    temp["price_score"] = 1-(temp["price"]/temp["price"].max())
    temp["rating_score"] = temp["rating"]/5
    temp["quality_score_norm"] = temp["quality_score"]/5
    temp["reliability_score_norm"] = temp["reliability_score"]/100
    temp["delivery_score"] = 1-(temp["average_delivery_time_min"]/temp["average_delivery_time_min"].max())

    temp["recommendation_score"]=(
        0.35*temp["distance_score"]+
        0.25*temp["price_score"]+
        0.15*temp["rating_score"]+
        0.10*temp["quality_score_norm"]+
        0.10*temp["reliability_score_norm"]+
        0.05*temp["delivery_score"]
    )

    return temp.sort_values(
        "recommendation_score",
        ascending=False
    ).head(top_n)


## Test Recommendation Engine

In [ ]:
recommend_supplier(
    latitude=18.5204,
    longitude=73.8567,
    ingredient="Potato"
)[[
    "supplier_name",
    "area",
    "price",
    "distance_km",
    "recommendation_score"
]]


# Key Observations

- The engine first filters suppliers by ingredient.
- Distance is computed dynamically using the vendor's live GPS coordinates.
- Suppliers outside their delivery radius are excluded.
- The trained scaler and K-Means model are reused for inference.
- Final recommendations are produced using a weighted business score rather than cluster labels alone.
- This notebook contains the core recommendation logic that will later be called from the backend used by the application.
